# A Small Language Model for Song Lyrics, Built from Scratch

This notebook trains a GPT-style small language model on song lyrics, using the course notebook's architecture. It saves every number, table, figure and sample the report needs.

**Data.** `spotify_millsongdata.csv` from Kaggle. **Only the `text` column (the lyrics) is used.**

| Report question | What this notebook produces |
|---|---|
| **Q1** Dataset challenges and corpus size | A cleaning pipeline with a statistics table for every step (songs, lines, words, tokens), plus before/after figures |
| **Q2** Fluent, domain-specific text? | Sample songs from fixed prompts, and measures of repetition, line structure, rhyme and verbatim copying, compared with real held-out songs |
| **Q3** Changes to the architecture or pipeline | Fixes to the original training loop, plus experiments on context length, tokenizer, model size and data cleaning |
| **Q4** What more compute or data would buy | A model-size and context-length trend, and a sampling-settings sweep |

**How to run**
1. In Colab, choose Runtime → Change runtime type → **GPU** (a T4 works; an L4 is faster).
2. Put `spotify_millsongdata.csv` in `My Drive/slm_lyrics/` (or upload it to the Colab working folder).
3. Run all cells. Everything is saved to Google Drive. Finished runs are skipped and an interrupted run resumes from its last checkpoint, so you can simply run all cells again after a disconnect.

**Outputs** (inside `My Drive/slm_lyrics/`)
```
data/      cleaned songs, train/validation split, tokenizers, token .bin files
runs/      one folder per run: config, loss log, best model, samples, metrics
figures/   every figure for the report
report/    data_stats.csv, results.csv, results_table.md, samples.md, key_numbers.md
```

## 0. Setup and configuration

In [ ]:
%pip install -q tiktoken tokenizers pronouncing langdetect

In [ ]:
import os, re, io, json, math, time, random, shutil, unicodedata, collections
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from contextlib import nullcontext

# ---------- where everything is saved ----------
USE_DRIVE = True
PROJECT_NAME = "slm_lyrics"
PROJECT_DIR = Path(PROJECT_NAME)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as e:
        print("Google Drive not available, saving locally:", e)
DATA_DIR, RUNS_DIR, FIG_DIR, REPORT_DIR = (PROJECT_DIR / d for d in ("data", "runs", "figures", "report"))
for d in (DATA_DIR, RUNS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CSV_NAME = "spotify_millsongdata.csv"
TEXT_COLUMN = "text"          # the only column used

# ---------- data cleaning ----------
MAX_LINE_REPEATS = 3          # keep at most this many copies of an identical line within a song
MIN_WORDS = 20                # drop songs shorter than this after cleaning
VAL_FRACTION = 0.05           # held-out songs (split by song, never inside a song)
DEBUG_MAX_SONGS = None        # e.g. 2000 for a quick test run; None = all songs

# ---------- training (same for every run unless the experiment changes it) ----------
SEED = 42
EPOCHS = 3                    # passes over the training tokens; fixes the budget for every run
TOKENS_PER_STEP = 16_384      # batch_size x block_size x grad_accum, kept constant across runs
GRAD_ACCUM = 4
LEARNING_RATE = 6e-4
MIN_LR = 6e-5                 # must be BELOW the learning rate (the original notebook had 5e-4 > 1e-4)
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.1
EVAL_INTERVAL = 250           # steps between evaluations
EVAL_ITERS = 50               # batches per evaluation (the original used 500, every 500 iterations)
CKPT_INTERVAL = 500

# ---------- evaluation ----------
N_GEN = 100                   # generated songs per run for the measurements
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.8
TOP_K = 50
NGRAM_COPY = 8                # a generated 8-word sequence found in the training data counts as copied
AUTO_DISCONNECT = True        # release the Colab GPU at the end to save compute credits

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else ("float16" if DEVICE == "cuda" else "float32")
PTDTYPE = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE]
CTX = nullcontext() if DEVICE == "cpu" else torch.amp.autocast(device_type="cuda", dtype=PTDTYPE)
print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| dtype:", DTYPE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU")

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10})
def savefig(fig, name):
    fig.savefig(FIG_DIR / name); print("saved", FIG_DIR / name); plt.show(); plt.close(fig)

## 1. Build the dataset (Q1)

The raw lyrics need several fixes before training. Each step is applied in order, and the table below records what it removed:

1. **Normalise text:** Windows line endings (`\r\n`) become `\n`, the two trailing spaces on every line are stripped, curly quotes become plain quotes, and runs of blank lines collapse to one blank line (kept as the stanza break).
2. **Remove annotations:** lines such as `[Chorus]`, `[Verse 1]`, `Chorus:`, `(Repeat chorus)`, `x2`, and inline `[?]` are not lyrics.
3. **Cap repeated lines:** a chorus repeated six times teaches the model to loop. An identical line is kept at most `MAX_LINE_REPEATS` times per song.
4. **Remove duplicate songs:** songs whose lyrics are identical after lower-casing and removing punctuation.
5. **Drop very short songs:** fewer than `MIN_WORDS` words after cleaning.

A second version without step 3 is also saved, for the data-cleaning experiment in Q3.

In [ ]:
def find_csv(name):
    for p in (Path(name), PROJECT_DIR / name, Path("/content") / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found. Put it in {PROJECT_DIR} or the Colab working folder.")

raw = pd.read_csv(find_csv(CSV_NAME), usecols=[TEXT_COLUMN])[TEXT_COLUMN].fillna("").astype(str)
if DEBUG_MAX_SONGS:
    raw = raw.sample(DEBUG_MAX_SONGS, random_state=SEED).reset_index(drop=True)
print(f"{len(raw):,} songs loaded from the '{TEXT_COLUMN}' column")

ANNOTATION_LINE = re.compile(
    r"^\s*[\[\(\{]?\s*(?:(?:repeat|pre-?chorus|chorus|verse|bridge|hook|outro|intro|refrain|instrumental|solo|"
    r"interlude|coda|break|spoken|x\s*\d+|\d+\s*x|\?)[\s:\-\.,0-9x()]*)+[\]\)\}]?\s*:?\s*$", re.I)
INLINE_BRACKET = re.compile(r"\[[^\]]*\]")
QUOTES = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u2026": "..."})

def normalise(s):
    s = unicodedata.normalize("NFKC", s).translate(QUOTES).replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", l).strip() for l in s.split("\n")]
    out, blank = [], False
    for l in lines:
        if l:
            out.append(l); blank = False
        elif out and not blank:
            out.append(""); blank = True
    while out and out[-1] == "":
        out.pop()
    return "\n".join(out)

def remove_annotations(s):
    out = []
    for l in s.split("\n"):
        if l and ANNOTATION_LINE.match(l):
            continue
        l = INLINE_BRACKET.sub("", l).strip() if "[" in l else l
        out.append(l)
    return normalise("\n".join(out))

def cap_repeats(s, k=MAX_LINE_REPEATS):
    seen, out = collections.Counter(), []
    for l in s.split("\n"):
        key = l.lower()
        if l:
            seen[key] += 1
            if seen[key] > k:
                continue
        out.append(l)
    return normalise("\n".join(out))

def song_stats(texts, label):
    lines = texts.apply(lambda s: sum(1 for l in s.splitlines() if l.strip()))
    words = texts.str.split().str.len()
    return dict(step=label, songs=len(texts), lines=int(lines.sum()), words=int(words.sum()), characters=int(texts.str.len().sum()))

def dedupe(texts):
    key = texts.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
    return texts[~key.duplicated()]

stats = [song_stats(raw, "0. raw")]
s1 = raw.apply(normalise);            stats.append(song_stats(s1, "1. normalise text"))
s2 = s1.apply(remove_annotations);   stats.append(song_stats(s2, "2. remove annotations"))
s3 = s2.apply(cap_repeats);          stats.append(song_stats(s3, f"3. cap repeated lines at {MAX_LINE_REPEATS}"))
s4 = dedupe(s3);                     stats.append(song_stats(s4, "4. remove duplicate songs"))
clean = s4[s4.str.split().str.len() >= MIN_WORDS].reset_index(drop=True)
stats.append(song_stats(clean, f"5. drop songs under {MIN_WORDS} words"))

# variant without the repetition cap (same songs kept, for the Q3 data experiment)
nocap = s2.loc[s4.index][s4.str.split().str.len() >= MIN_WORDS].reset_index(drop=True)

data_stats = pd.DataFrame(stats)
data_stats["characters_removed_%"] = (100 * (1 - data_stats.characters / data_stats.characters.iloc[0])).round(2)
data_stats.to_csv(REPORT_DIR / "data_stats.csv", index=False)
data_stats

In [ ]:
def repeated_line_share(s):
    L = [l.lower() for l in s.split("\n") if l]
    return 1 - len(set(L)) / max(1, len(L))

def annotation_lines(s):
    return sum(1 for l in normalise(s).split("\n") if l and ANNOTATION_LINE.match(l))

rng = np.random.default_rng(SEED)
lang_sample = clean.sample(min(1000, len(clean)), random_state=SEED)
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    langs = collections.Counter()
    for s in lang_sample:
        try: langs[detect(s[:600])] += 1
        except Exception: langs["unknown"] += 1
    english_share = langs.get("en", 0) / len(lang_sample)
except Exception as e:
    print("language check skipped:", e); langs, english_share = {}, float("nan")

summary = {
    "songs_raw": len(raw), "songs_clean": len(clean),
    "annotation_lines_removed": int(raw.apply(annotation_lines).sum()),
    "duplicate_songs_removed": int(len(s3) - len(s4)),
    "repeated_line_share_before_cap": float(s2.apply(repeated_line_share).mean()),
    "repeated_line_share_after_cap": float(clean.apply(repeated_line_share).mean()),
    "english_share_sample": english_share, "languages_sample": dict(collections.Counter(langs).most_common(5)),
    "words_per_song_median": float(clean.str.split().str.len().median()),
    "lines_per_song_mean": float(data_stats.lines.iloc[-1] / len(clean)),
}
json.dump(summary, open(REPORT_DIR / "dataset_summary.json", "w"), indent=2)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.6), gridspec_kw={"wspace": 0.6})
ax[0].hist(raw.str.split().str.len().clip(upper=800), bins=60, alpha=.6, label="raw")
ax[0].hist(clean.str.split().str.len().clip(upper=800), bins=60, alpha=.6, label="clean")
ax[0].set_xlabel("words per song"); ax[0].set_ylabel("songs"); ax[0].legend()
ax[1].hist(s2.apply(repeated_line_share), bins=40, alpha=.6, label="before cap")
ax[1].hist(clean.apply(repeated_line_share), bins=40, alpha=.6, label=f"after cap ({MAX_LINE_REPEATS})")
ax[1].set_xlabel("share of lines that repeat within the song"); ax[1].legend()
d = data_stats.set_index("step")
ax[2].barh(d.index[::-1], d["characters_removed_%"][::-1], color="gray")
ax[2].set_xlabel("cumulative % of characters removed")
fig.suptitle("Q1: lyrics dataset before and after cleaning")
savefig(fig, "fig_q1_data_cleaning.png")
pd.Series(summary)

## 2. Split and tokenize

Songs are split **by song** into training and validation sets, so the model is always validated on songs it never saw. Each song ends with an end-of-text token, so the model learns where a song stops.

Two tokenizers are compared:
* **`gpt2`**: the course notebook's GPT-2 tokenizer (50,257 tokens).
* **`bpe8k`**: a byte-level BPE tokenizer with 8,192 tokens, trained on the training lyrics only. A smaller vocabulary shrinks the embedding table, which is most of the original model's parameters.

Because the tokenizers split text differently, loss per token is not comparable between them. The notebook therefore also reports **bits per character**, which is.

In [ ]:
import tiktoken
from tokenizers import Tokenizer, ByteLevelBPETokenizer

DATASETS = {"clean": clean, "nocap": nocap}
perm = np.random.default_rng(SEED).permutation(len(clean))
n_val = int(round(len(clean) * VAL_FRACTION))
VAL_IDX, TRAIN_IDX = np.sort(perm[:n_val]), np.sort(perm[n_val:])
print(f"train songs: {len(TRAIN_IDX):,}   validation songs: {len(VAL_IDX):,}")

class Tok:
    def __init__(self, name, encode, decode, eot, vocab_size):
        self.name, self.encode, self.decode, self.eot, self.vocab_size = name, encode, decode, eot, vocab_size

_TOK_CACHE = {}
def get_tokenizer(name):
    if name in _TOK_CACHE:
        return _TOK_CACHE[name]
    if name == "gpt2":
        enc = tiktoken.get_encoding("gpt2")
        tok = Tok("gpt2", enc.encode_ordinary, enc.decode, enc.eot_token, enc.n_vocab)
    elif name.startswith("bpe"):
        size = int(name[3:].replace("k", "")) * 1024
        path = DATA_DIR / f"tokenizer_{name}.json"
        if not path.exists():
            t = ByteLevelBPETokenizer()
            t.train_from_iterator(list(clean.iloc[TRAIN_IDX]), vocab_size=size, min_frequency=2,
                                  special_tokens=["<|endoftext|>"], show_progress=False)
            t.save(str(path))
        hf = Tokenizer.from_file(str(path))
        tok = Tok(name, lambda s, hf=hf: hf.encode(s).ids, lambda ids, hf=hf: hf.decode(ids),
                  hf.token_to_id("<|endoftext|>"), hf.get_vocab_size())
    else:
        raise ValueError(name)
    _TOK_CACHE[name] = tok
    return tok

def prepare_bins(dataset, tok_name):
    # Tokenize one dataset version with one tokenizer and write train/val .bin files (cached on Drive).
    out = DATA_DIR / f"{dataset}_{tok_name}"
    meta_p = out / "meta.json"
    if meta_p.exists():
        return json.load(open(meta_p))
    out.mkdir(parents=True, exist_ok=True)
    tok, texts = get_tokenizer(tok_name), DATASETS[dataset]
    meta = {"dataset": dataset, "tokenizer": tok_name, "vocab_size": tok.vocab_size, "eot": tok.eot}
    for split, idx in (("train", TRAIN_IDX), ("val", VAL_IDX)):
        ids = []
        for s in texts.iloc[idx]:
            ids.extend(tok.encode(s)); ids.append(tok.eot)
        arr = np.array(ids, dtype=np.uint16)
        arr.tofile(out / f"{split}.bin")
        meta[f"{split}_tokens"] = int(len(arr))
        meta[f"{split}_chars"] = int(sum(len(s) + 1 for s in texts.iloc[idx]))
    json.dump(meta, open(meta_p, "w"), indent=2)
    return meta

for ds, tk in (("clean", "gpt2"), ("clean", "bpe8k"), ("nocap", "gpt2")):
    m = prepare_bins(ds, tk)
    print(f"{ds:6s} {tk:6s}  train {m['train_tokens']:>11,} tokens  val {m['val_tokens']:>9,} tokens  "
          f"({m['train_chars'] / m['train_tokens']:.2f} characters per token)")

## 3. Model architecture

The architecture is the course notebook's GPT, unchanged: token and position embeddings, pre-norm transformer blocks with causal self-attention and a 4× MLP, and weight tying between the embedding and output layers. Only `generate` is extended: it stops a song at the end-of-text token and takes a random-number generator, so samples are reproducible.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.flash = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                 .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                               dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            y = self.attn_dropout(F.softmax(att, dim=-1)) @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            return logits, F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return self.lm_head(x[:, [-1], :]), None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eot=None, generator=None):
        # Batched sampling. Sequences that produce `eot` stop growing (padded with eot).
        done = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :].float() / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1, generator=generator)
            if eot is not None:
                nxt[done] = eot
                done |= nxt.squeeze(1) == eot
            idx = torch.cat((idx, nxt), dim=1)
            if eot is not None and bool(done.all()):
                break
        return idx

def count_params(model):
    return sum(p.numel() for p in model.parameters())   # tied weights counted once

## 4. Training

Changes from the original training loop, kept for every run:

| Original | Here | Why |
|---|---|---|
| `min_lr = 5e-4` above `learning_rate = 1e-4` | 6e-4 decaying to 6e-5 | The original learning rate rose during training instead of decaying |
| 20,000 iterations × 32 accumulation = 625 weight updates | A fixed number of passes over the data (`EPOCHS`), with a constant 16,384 tokens per update | Every run sees the same data the same number of times, whatever its context length |
| Scheduler stepped every micro-batch | Learning rate set once per weight update | Warm-up and decay follow real updates |
| 500 evaluation batches every 500 iterations | 50 batches every 250 updates | Evaluation took more time than training |
| Model saved in the Colab session only | Checkpoints and loss log saved to Drive | Survives disconnects; runs resume |

In [ ]:
@dataclass
class RunCfg:
    name: str
    group: str = "Q3"
    dataset: str = "clean"
    tokenizer: str = "gpt2"
    block_size: int = 128
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    note: str = ""

    @property
    def dir(self):
        return RUNS_DIR / self.name

def get_batch(meta_dir, split, block_size, batch_size, gen):
    data = np.memmap(meta_dir / f"{split}.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size - 1, (batch_size,), generator=gen).tolist()
    x = torch.from_numpy(np.stack([data[i:i + block_size].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([data[i + 1:i + 1 + block_size].astype(np.int64) for i in ix]))
    if DEVICE == "cuda":
        return x.pin_memory().to(DEVICE, non_blocking=True), y.pin_memory().to(DEVICE, non_blocking=True)
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(model, meta_dir, block_size, batch_size):
    model.eval()
    gen = torch.Generator().manual_seed(1234)          # same evaluation batches every time
    out = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(EVAL_ITERS):
            X, Y = get_batch(meta_dir, split, block_size, batch_size, gen)
            with CTX:
                losses.append(model(X, Y)[1].item())
        out[split] = float(np.mean(losses))
    model.train()
    return out

def build_model(cfg, vocab_size):
    return GPT(GPTConfig(block_size=cfg.block_size, vocab_size=vocab_size, n_layer=cfg.n_layer,
                         n_head=cfg.n_head, n_embd=cfg.n_embd, dropout=cfg.dropout, bias=True))

def lr_at(step, total):
    warm = min(WARMUP_STEPS, max(1, total // 10))
    if step < warm:
        return LEARNING_RATE * (step + 1) / warm
    p = (step - warm) / max(1, total - warm)
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * min(1.0, p)))

def train_run(cfg: RunCfg):
    run_dir = cfg.dir
    if (run_dir / "train_info.json").exists():
        print(f"[skip] {cfg.name} already trained"); return json.load(open(run_dir / "train_info.json"))
    run_dir.mkdir(parents=True, exist_ok=True)
    meta = prepare_bins(cfg.dataset, cfg.tokenizer)
    meta_dir = DATA_DIR / f"{cfg.dataset}_{cfg.tokenizer}"
    batch_size = TOKENS_PER_STEP // (cfg.block_size * GRAD_ACCUM)
    steps = math.ceil(EPOCHS * meta["train_tokens"] / TOKENS_PER_STEP)
    json.dump(asdict(cfg) | {"batch_size": batch_size, "steps": steps}, open(run_dir / "config.json", "w"), indent=2)

    torch.manual_seed(SEED)
    model = build_model(cfg, meta["vocab_size"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == "float16"))
    n_params = count_params(model)
    start, log, best, elapsed = 0, [], float("inf"), 0.0
    ck = run_dir / "ckpt.pt"
    if ck.exists():
        state = torch.load(ck, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"]); opt.load_state_dict(state["opt"]); scaler.load_state_dict(state["scaler"])
        start, log, best, elapsed = state["step"], state["log"], state["best"], state["elapsed"]
        print(f"[resume] {cfg.name} from step {start}")
    print(f"[train] {cfg.name}: {n_params / 1e6:.1f}M params, {steps} updates of {TOKENS_PER_STEP:,} tokens "
          f"({EPOCHS} passes over {meta['train_tokens']:,} tokens)")
    gen = torch.Generator().manual_seed(SEED * 1000 + start)
    t0 = time.time()
    model.train()
    for step in range(start, steps):
        if step % EVAL_INTERVAL == 0 or step == steps - 1:
            l = estimate_loss(model, meta_dir, cfg.block_size, batch_size)
            log.append(dict(step=step, tokens=step * TOKENS_PER_STEP, train_loss=l["train"], val_loss=l["val"], lr=lr_at(step, steps)))
            print(f"   step {step:5d}/{steps}  train {l['train']:.3f}  val {l['val']:.3f}  lr {lr_at(step, steps):.2e}")
            if l["val"] < best:
                best = l["val"]; torch.save(model.state_dict(), run_dir / "best_model.pt")
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps)
        for _ in range(GRAD_ACCUM):
            X, Y = get_batch(meta_dir, "train", cfg.block_size, batch_size, gen)
            with CTX:
                loss = model(X, Y)[1] / GRAD_ACCUM
            scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step == start + 20:
            rate = 20 / (time.time() - t0)
            print(f"   {rate:.2f} updates/s -> about {(steps - step) / rate / 60:.0f} min for this run")
        if (step + 1) % CKPT_INTERVAL == 0 and step + 1 < steps:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), step=step + 1,
                            log=log, best=best, elapsed=elapsed + time.time() - t0), ck)
    elapsed += time.time() - t0
    pd.DataFrame(log).to_csv(run_dir / "loss_log.csv", index=False)
    info = dict(name=cfg.name, params_M=n_params / 1e6, steps=steps, tokens_seen=steps * TOKENS_PER_STEP,
                train_minutes=elapsed / 60, best_val_loss=best,
                gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
    json.dump(info, open(run_dir / "train_info.json", "w"), indent=2)
    if ck.exists():
        ck.unlink()
    del model, opt
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return info

## 5. Evaluation (Q2)

Each run generates `N_GEN` songs from the start-of-song token, plus completions of fixed prompts. The same random seed is used for every run, so samples are directly comparable.

| Measure | Meaning | Real songs give |
|---|---|---|
| **Validation loss / perplexity** | How well the model predicts unseen songs (per token) | — |
| **Bits per character** | The same, but comparable across tokenizers | lower is better |
| **Distinct-2** | Share of word pairs in a song that are unique; low means looping | reference value |
| **Repeated-line share** | Share of a song's lines that repeat an earlier line | reference value |
| **Lines per song, words per line** | Whether the output is shaped like lyrics | reference value |
| **Rhyme rate** | Share of lines whose last word rhymes with one of the next two lines (CMU pronouncing dictionary) | reference value |
| **Copy rate** | Share of generated 8-word sequences found word-for-word in the training lyrics | near 0 |
| **Longest copied run** | Longest stretch of words copied from one training song | short |

The prompts below are original lines written for this project, not lyrics from the dataset.

In [ ]:
import pronouncing

PROMPTS = [
    "I've been driving all night",
    "Baby, don't you leave me now",
    "When the summer rain is falling",
    "I remember the day we met",
    "Dancing under city lights",
    "My heart is a lonely highway",
    "Hold on, hold on",
    "",                                  # empty prompt = start of a new song
]

WORD_RE = re.compile(r"[a-z']+")
def words_of(s):
    return WORD_RE.findall(s.lower())

def lyric_lines(s):
    return [l.strip() for l in s.split("\n") if l.strip()]

def distinct2(s):
    w = words_of(s); bg = list(zip(w, w[1:]))
    return len(set(bg)) / len(bg) if bg else float("nan")

def rhyming_parts(word):
    return {pronouncing.rhyming_part(p) for p in pronouncing.phones_for_word(word)}

def rhyme_rate(s):
    stanzas = [lyric_lines(st) for st in s.split("\n\n")]
    hits = total = 0
    for L in stanzas:
        last = [words_of(l)[-1] if words_of(l) else "" for l in L]
        parts = [rhyming_parts(w) if w else set() for w in last]
        for i in range(len(L) - 1):
            if not parts[i]:
                continue
            total += 1
            if any(j < len(L) and last[j] != last[i] and parts[i] & parts[j] for j in (i + 1, i + 2)):
                hits += 1
    return hits / total if total else float("nan")

def text_metrics(songs):
    rows = []
    for s in songs:
        L = lyric_lines(s)
        rows.append(dict(distinct2=distinct2(s), repeated_line_share=repeated_line_share(s) if L else float("nan"),
                         lines=len(L), words_per_line=np.mean([len(words_of(l)) for l in L]) if L else float("nan"),
                         rhyme_rate=rhyme_rate(s), empty=len(L) < 3))
    d = pd.DataFrame(rows)
    return {k: float(d[k].mean()) for k in d.columns}

class NgramIndex:
    # Hashes of every word 8-gram in the training lyrics, for the copy check.
    def __init__(self, texts, n=NGRAM_COPY):
        self.n, self.vocab = n, {}
        rs = np.random.default_rng(0)
        self.mult = rs.integers(1, 2**62, size=n, dtype=np.uint64) | np.uint64(1)
        chunks = []
        for s in texts:
            ids = np.array([self.vocab.setdefault(w, len(self.vocab)) for w in words_of(s)], dtype=np.uint64)
            if len(ids) >= n:
                chunks.append(self._hash(ids))
        self.hashes = np.unique(np.concatenate(chunks))
    def _hash(self, ids):
        win = np.lib.stride_tricks.sliding_window_view(ids, self.n)
        with np.errstate(over="ignore"):
            return (win * self.mult).sum(axis=1)
    def copied(self, s):
        # returns (share of 8-grams copied, longest copied run in words)
        ids = [self.vocab.get(w) for w in words_of(s)]
        if len(ids) < self.n:
            return float("nan"), 0
        known = np.array([i if i is not None else -1 for i in ids], dtype=np.int64)
        win_ok = np.lib.stride_tricks.sliding_window_view(known >= 0, self.n).all(axis=1)
        h = self._hash(np.where(known >= 0, known, 0).astype(np.uint64))
        hit = win_ok & np.isin(h, self.hashes, assume_unique=False)
        longest, run = 0, 0
        for x in hit:
            run = run + 1 if x else 0
            longest = max(longest, run)
        return float(hit.mean()), (longest + self.n - 1) if longest else 0

_NGRAM = {}
def ngram_index(dataset="clean"):
    if dataset not in _NGRAM:
        t0 = time.time()
        _NGRAM[dataset] = NgramIndex(DATASETS[dataset].iloc[TRAIN_IDX])
        print(f"8-gram index of the training lyrics: {len(_NGRAM[dataset].hashes):,} entries ({time.time() - t0:.0f}s)")
    return _NGRAM[dataset]

def load_model(cfg):
    meta = prepare_bins(cfg.dataset, cfg.tokenizer)
    model = build_model(cfg, meta["vocab_size"]).to(DEVICE)
    model.load_state_dict(torch.load(cfg.dir / "best_model.pt", map_location=DEVICE))
    return model.eval(), get_tokenizer(cfg.tokenizer), meta

def sample_songs(model, tok, prompts, temperature=TEMPERATURE, top_k=TOP_K, seed=SEED, max_new=MAX_NEW_TOKENS, batch=25):
    # Prompts with the same token length are generated together in batches.
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    enc = [[tok.eot] + tok.encode(p) for p in prompts]
    groups = collections.defaultdict(list)
    for i, e in enumerate(enc):
        groups[len(e)].append(i)
    texts = [None] * len(prompts)
    for L in sorted(groups):
        idxs = groups[L]
        for j in range(0, len(idxs), batch):
            chunk = idxs[j:j + batch]
            x = torch.tensor([enc[i] for i in chunk], dtype=torch.long, device=DEVICE)
            with CTX:
                y = model.generate(x, max_new, temperature=temperature, top_k=top_k, eot=tok.eot, generator=g)
            for i, row in zip(chunk, y[:, L:].tolist()):
                if tok.eot in row:
                    row = row[:row.index(tok.eot)]
                texts[i] = prompts[i] + tok.decode(row)
    return texts

def reference_metrics():
    p = REPORT_DIR / "reference_real_songs.json"
    if p.exists():
        return json.load(open(p))
    real = list(clean.iloc[VAL_IDX].sample(min(300, len(VAL_IDX)), random_state=SEED))
    m = text_metrics(real)
    json.dump(m, open(p, "w"), indent=2)
    return m

def evaluate_run(cfg: RunCfg, force=False):
    mpath = cfg.dir / "metrics.json"
    if mpath.exists() and not force:
        return json.load(open(mpath))
    print(f"[eval] {cfg.name}")
    model, tok, _ = load_model(cfg)
    # Always score on the CLEANED validation songs, so runs trained on other data versions stay comparable
    meta = prepare_bins("clean", cfg.tokenizer)
    meta_dir = DATA_DIR / f"clean_{cfg.tokenizer}"
    batch_size = TOKENS_PER_STEP // (cfg.block_size * GRAD_ACCUM)
    # full validation loss
    val = np.memmap(meta_dir / "val.bin", dtype=np.uint16, mode="r")
    losses, B = [], cfg.block_size
    with torch.no_grad():
        for i in range(0, len(val) - B - 1, B * batch_size):
            starts = [j for j in range(i, min(i + B * batch_size, len(val) - B - 1), B)]
            x = torch.from_numpy(np.stack([val[j:j + B].astype(np.int64) for j in starts])).to(DEVICE)
            y = torch.from_numpy(np.stack([val[j + 1:j + 1 + B].astype(np.int64) for j in starts])).to(DEVICE)
            with CTX:
                losses.append((model(x, y)[1].item(), len(starts)))
    val_loss = sum(l * n for l, n in losses) / sum(n for _, n in losses)
    bpc = val_loss * meta["val_tokens"] / meta["val_chars"] / math.log(2)
    # generation
    t0 = time.time()
    uncond = sample_songs(model, tok, [""] * N_GEN)
    gen_sec = (time.time() - t0) / N_GEN
    prompted = sample_songs(model, tok, PROMPTS, seed=SEED + 1)
    json.dump({"unconditional": uncond, "prompted": dict(zip(PROMPTS, prompted))}, open(cfg.dir / "samples.json", "w"), indent=1)
    tm = text_metrics(uncond)
    idx = ngram_index("clean")
    cp = [idx.copied(s) for s in uncond]
    info = json.load(open(cfg.dir / "train_info.json"))
    metrics = (asdict(cfg) | info | dict(val_loss=val_loss, val_perplexity=math.exp(val_loss), val_bpc=bpc,
               copy_rate=float(np.nanmean([c[0] for c in cp])), longest_copy_words=int(max(c[1] for c in cp)),
               songs_with_long_copy=float(np.mean([c[1] >= 15 for c in cp])), sec_per_song=gen_sec)
               | {f"gen_{k}": v for k, v in tm.items()})
    json.dump(metrics, open(mpath, "w"), indent=2)
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return metrics

## 6. Experiment plan (Q3, Q4)

Each run changes **one thing** from the baseline. All runs see the training data `EPOCHS` times with the same number of tokens per update.

| Run | Change from baseline | Question it answers |
|---|---|---|
| `baseline` | Course model (6 layers, 6 heads, 384 wide, 128-token context, GPT-2 tokenizer) on the cleaned data | Reference |
| `ctx256` | Context 256 tokens | Does seeing more of the song (verse and chorus) help? |
| `bpe8k` | Own 8,192-token tokenizer | Does a lyrics-specific, smaller vocabulary help? |
| `small` | 4 layers, 4 heads, 256 wide | Model-size trend (Q4) |
| `large` | 8 layers, 8 heads, 512 wide (about 51M parameters, the notebook's stated 50–60M target) | Model-size trend (Q4) |
| `nocap` | Training data without the repeated-line cap | Does the cleaning step reduce looping? |

The first run prints its speed and estimated time. On an L4 each run should take roughly 5–20 minutes; check the printed estimate.

In [ ]:
EXPERIMENTS = [
    RunCfg("baseline", group="baseline", note="course model on cleaned lyrics"),
    RunCfg("ctx256", block_size=256, note="context 128 -> 256"),
    RunCfg("bpe8k", tokenizer="bpe8k", note="own 8,192-token BPE tokenizer"),
    RunCfg("small", n_layer=4, n_head=4, n_embd=256, note="smaller model"),
    RunCfg("large", n_layer=8, n_head=8, n_embd=512, note="larger model"),
    RunCfg("nocap", dataset="nocap", note="no repeated-line cap"),
]
RUN_ONLY = None    # e.g. ["baseline", "ctx256"] to run part of the plan
FORCE_EVAL = ["nocap"]   # re-score these runs (nocap was first scored on its own, uncapped validation set)

reference = reference_metrics()
for cfg in EXPERIMENTS:
    if RUN_ONLY and cfg.name not in RUN_ONLY:
        continue
    train_run(cfg)
    m = evaluate_run(cfg, force=cfg.name in FORCE_EVAL and not (cfg.dir / "metrics_v2.flag").exists())
    if cfg.name in FORCE_EVAL:
        (cfg.dir / "metrics_v2.flag").touch()
    print(f"   -> val loss {m['val_loss']:.3f}  bits/char {m['val_bpc']:.3f}  distinct-2 {m['gen_distinct2']:.2f}  "
          f"copy {m['copy_rate']:.1%}  rhyme {m['gen_rhyme_rate']:.2f}")

## 7. Results

In [ ]:
rows = [json.load(open(p)) for p in sorted(RUNS_DIR.glob("*/metrics.json"))]
results = pd.DataFrame(rows)
order = [c.name for c in EXPERIMENTS if c.name in set(results.name)]
results = results.set_index("name").loc[order].reset_index()
results.to_csv(REPORT_DIR / "results.csv", index=False)

cols = ["name", "note", "params_M", "val_loss", "val_perplexity", "val_bpc", "gen_distinct2", "gen_repeated_line_share",
        "gen_lines", "gen_words_per_line", "gen_rhyme_rate", "copy_rate", "longest_copy_words", "train_minutes"]
table = results[cols].round(3)
ref_row = pd.DataFrame([{"name": "real songs (validation)", "gen_distinct2": reference["distinct2"],
                         "gen_repeated_line_share": reference["repeated_line_share"], "gen_lines": reference["lines"],
                         "gen_words_per_line": reference["words_per_line"], "gen_rhyme_rate": reference["rhyme_rate"]}]).round(3)
table_with_ref = pd.concat([table, ref_row], ignore_index=True)
(REPORT_DIR / "results_table.md").write_text(table_with_ref.to_markdown(index=False))
table_with_ref

In [ ]:
# Validation loss curves, in bits per character so all runs (both tokenizers) are comparable
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name in order:
    log = pd.read_csv(RUNS_DIR / name / "loss_log.csv")
    meta = prepare_bins(results.set_index("name").loc[name, "dataset"], results.set_index("name").loc[name, "tokenizer"])
    k = meta["val_tokens"] / meta["val_chars"] / math.log(2)
    ax[0].plot(log.step / log.step.max() * EPOCHS, log.val_loss * k, label=name)
    if name == "baseline":
        ax[1].plot(log.step / log.step.max() * EPOCHS, log.train_loss, label="train")
        ax[1].plot(log.step / log.step.max() * EPOCHS, log.val_loss, label="validation")
ax[0].set_title("nocap is scored on its own (uncapped) validation data here; see the table for the comparable value", fontsize=8)
ax[0].set_xlabel("passes over the training data"); ax[0].set_ylabel("validation bits per character"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_xlabel("passes over the training data"); ax[1].set_ylabel("loss per token (baseline)"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("baseline: train vs validation (a widening gap = overfitting)")
savefig(fig, "fig_loss_curves.png")

In [ ]:
# Q3 comparison: each change against the baseline
metrics_plot = [("val_bpc", "validation bits/char (lower better)"), ("gen_distinct2", "distinct-2 (higher = less looping)"),
                ("gen_rhyme_rate", "rhyme rate"), ("copy_rate", "copy rate (8-word sequences)")]
ref_map = {"gen_distinct2": reference["distinct2"], "gen_rhyme_rate": reference["rhyme_rate"]}
fig, axes = plt.subplots(1, len(metrics_plot), figsize=(4.2 * len(metrics_plot), 3.6))
for ax, (col, lab) in zip(axes, metrics_plot):
    vals = results.set_index("name").loc[order, col]
    ax.bar(order, vals, color=["#444" if n == "baseline" else "#4C72B0" for n in order])
    if col in ref_map:
        ax.axhline(ref_map[col], color="#C44E52", ls="--", lw=1, label="real songs"); ax.legend(fontsize=8)
    ax.set_title(lab, fontsize=9); ax.tick_params(axis="x", rotation=40)
fig.suptitle("Q3: effect of each change")
savefig(fig, "fig_q3_comparison.png")

# Q4 trend: model size and context length
r = results.set_index("name")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
size_runs = [n for n in ("small", "baseline", "large") if n in r.index]
ax[0].plot(r.loc[size_runs, "params_M"], r.loc[size_runs, "val_bpc"], "o-")
for n in size_runs:
    ax[0].annotate(n, (r.loc[n, "params_M"], r.loc[n, "val_bpc"]), textcoords="offset points", xytext=(5, 5))
ax[0].set_xlabel("parameters (millions)"); ax[0].set_ylabel("validation bits per character"); ax[0].grid(alpha=.3)
ctx_runs = [n for n in ("baseline", "ctx256") if n in r.index]
ax[1].bar([str(int(r.loc[n, "block_size"])) for n in ctx_runs], r.loc[ctx_runs, "val_bpc"], color="#4C72B0")
ax[1].set_xlabel("context length (tokens)"); ax[1].set_ylabel("validation bits per character")
lo = r.loc[ctx_runs, "val_bpc"].min(); ax[1].set_ylim(lo * 0.97, r.loc[ctx_runs, "val_bpc"].max() * 1.01)
fig.suptitle("Q4: what more model or context buys")
savefig(fig, "fig_q4_scaling.png")

In [ ]:
# Sampling settings sweep on the best run (lowest bits per character): fluency vs. variety vs. copying
best_name = results.loc[results.val_bpc.idxmin(), "name"]
best_cfg = next(c for c in EXPERIMENTS if c.name == best_name)
sweep_p = REPORT_DIR / "sampling_sweep.csv"
if sweep_p.exists():
    sweep = pd.read_csv(sweep_p)
else:
    model, tok, _ = load_model(best_cfg)
    rows = []
    for temp, k in [(0.6, 50), (0.8, 50), (1.0, 50), (1.2, 50), (1.0, None)]:
        songs = sample_songs(model, tok, [""] * min(60, N_GEN), temperature=temp, top_k=k, seed=SEED + 7)
        cp = [ngram_index().copied(s) for s in songs]
        rows.append(dict(temperature=temp, top_k=k if k else "none", **text_metrics(songs),
                         copy_rate=float(np.nanmean([c[0] for c in cp]))))
    sweep = pd.DataFrame(rows); sweep.to_csv(sweep_p, index=False)
    del model
print("best run:", best_name)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
lab = [f"T={t}\nk={k}" for t, k in zip(sweep.temperature, sweep.top_k)]
for a, (col, title) in zip(ax, [("distinct2", "distinct-2"), ("rhyme_rate", "rhyme rate"), ("copy_rate", "copy rate")]):
    a.bar(lab, sweep[col], color="#4C72B0"); a.set_title(title)
    if col in ("distinct2", "rhyme_rate"):
        a.axhline(reference[col], color="#C44E52", ls="--", lw=1, label="real songs"); a.legend(fontsize=8)
fig.suptitle(f"Q2: effect of sampling temperature (T) and top-k, run '{best_name}'")
savefig(fig, "fig_q2_sampling.png")
sweep.round(3)

In [ ]:
# Sample outputs for the submission (original prompts only)
lines_md = [f"# Sample outputs\n\nModel run: **{best_name}** (lowest validation bits per character). "
            f"Sampling: temperature {TEMPERATURE}, top-k {TOP_K}. Prompts are original lines written for this project.\n"]
for name in [best_name] + [n for n in order if n != best_name]:
    smp = json.load(open(RUNS_DIR / name / "samples.json"))
    lines_md.append(f"\n## Run: {name}\n")
    items = list(smp["prompted"].items()) if name == best_name else list(smp["prompted"].items())[:2]
    for p, txt in items:
        lines_md.append(f"\n**Prompt:** {p if p else '(empty: start of a new song)'}\n\n```\n{txt.strip()}\n```\n")
(REPORT_DIR / "samples.md").write_text("".join(lines_md))
print("".join(lines_md[:4]))

In [ ]:
b = results.set_index("name")
base = b.loc["baseline"] if "baseline" in b.index else b.iloc[0]
kn = [f"## Data\n- {summary['songs_raw']:,} songs -> {summary['songs_clean']:,} after cleaning; "
      f"{summary['annotation_lines_removed']:,} annotation lines and {summary['duplicate_songs_removed']:,} duplicate songs removed",
      f"- Repeated-line share {summary['repeated_line_share_before_cap']:.1%} -> {summary['repeated_line_share_after_cap']:.1%} after the cap",
      f"- Training corpus (GPT-2 tokens): {prepare_bins('clean', 'gpt2')['train_tokens']:,}; validation {prepare_bins('clean', 'gpt2')['val_tokens']:,}",
      "\n## Runs (validation bits/char, distinct-2, rhyme rate, copy rate)"]
for n in order:
    x = b.loc[n]
    kn.append(f"- {n} ({x.params_M:.1f}M params): bpc {x.val_bpc:.3f}, perplexity {x.val_perplexity:.1f}, "
              f"distinct-2 {x.gen_distinct2:.2f}, rhyme {x.gen_rhyme_rate:.2f}, copy {x.copy_rate:.2%}, longest copy {int(x.longest_copy_words)} words")
kn.append(f"- real songs: distinct-2 {reference['distinct2']:.2f}, rhyme {reference['rhyme_rate']:.2f}, "
          f"repeated lines {reference['repeated_line_share']:.2f}, lines/song {reference['lines']:.0f}")
kn.append(f"\n## Best run: {best_name}")
(REPORT_DIR / "key_numbers.md").write_text("\n".join(kn))
print("\n".join(kn))

In [ ]:
# Release the GPU so it stops using compute credits (results are already on Drive)
if AUTO_DISCONNECT:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("not on Colab, nothing to disconnect:", e)